In [22]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Imports
# ═══════════════════════════════════════════════════════════════
from the_well.data.normalization import ZScoreNormalization
import sys, os, datetime, fsspec, torch, munch, toml, yaml
from the_well.data import WellDataModule
from torch.utils.data import DataLoader
import torch.nn.functional as F
import pytorch_lightning as pl
import shutil

PROJECT_ROOT = "/Users/gabrieleaccarino/Workspace/waveletflux"
sys.path.insert(0, PROJECT_ROOT)

from waveletflux.utils.configs import WaveletConfig, VelocityNetConfig, FlowMatchingConfig, OptimizerConfig
from waveletflux.utils.well_data import WellDataModuleLightning
from waveletflux.utils.augmentations import compose_augmentation, compose_transformation
from waveletflux.utils.experiment import ExperimentManager
from waveletflux.modules.fourier_fm import FourierFlowModule, FourierConfig
from waveletflux.models.wavelet_unet import WaveletFlowUNet

print("Imports OK | MPS:", torch.backends.mps.is_available())
print(f"MPS allocated : {torch.mps.current_allocated_memory() / 1e9:.2f} GB")
print(f"MPS reserved  : {torch.mps.driver_allocated_memory()  / 1e9:.2f} GB")

Imports OK | MPS: True
MPS allocated : 0.00 GB
MPS reserved  : 1.50 GB


In [23]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Config
# ═══════════════════════════════════════════════════════════════
CONFIG_PATH = f"{PROJECT_ROOT}/configs/rayleigh_benard/FourierFM_rayleigh_benard_16modes_v4.0.toml"
#CONFIG_PATH = f"{PROJECT_ROOT}/configs/turbulent_radiative_layer_2D/WaveletFlow_turbulent_radiative_layer_2D_3scales_db2_v0.0.toml"
well_base_path    = f"/Users/gabrieleaccarino/Workspace/waveletflux/data/datasets"
well_dataset_name = 'rayleigh_benard'
#well_dataset_name = 'turbulent_radiative_layer_2D'
stats_path        = f"{well_base_path}/{well_dataset_name}/stats.yaml"
experiments_path  = f"{PROJECT_ROOT}/experiments/{well_dataset_name}"

config = munch.munchify(toml.load(CONFIG_PATH))

# Override for local MPS run
config.lightning.trainer.accelerator      = "mps"
config.lightning.trainer.strategy         = "auto"
config.lightning.trainer.precision        = "32"
config.lightning.trainer.matmul_precision = "high"
#config.train.batch_size                   = 1  # default from the config file
#config.train.epochs                       = 2  # default from the config file

pl.seed_everything(config.random.seed)

lt_train = config.lightning.trainer
torch.set_float32_matmul_precision(lt_train.matmul_precision)

# Verify context arithmetic BEFORE anything is built
ctx_ch = config.velocity_net.params.context_in_channels * config.velocity_net.params.num_context_frames
print(f"context_in_channels : {config.velocity_net.params.context_in_channels}")
print(f"num_context_frames  : {config.velocity_net.params.num_context_frames}")
print(f"ctx_ch              : {ctx_ch}   ← must be 12")
assert ctx_ch == 12, f"ctx_ch={ctx_ch}, expected 12"
print("✓ Config OK")

Seed set to 1234


context_in_channels : 4
num_context_frames  : 3
ctx_ch              : 12   ← must be 12
✓ Config OK


In [24]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Typed configs
# ═══════════════════════════════════════════════════════════════
fourier_cfg       = FourierConfig.from_config(config.fourier)
velocity_net_cfg  = VelocityNetConfig.from_config(config.velocity_net)
flow_matching_cfg = FlowMatchingConfig.from_config(config.flow_matching, config.sampling)
optimizer_cfg     = OptimizerConfig.from_config(config.optimizer, config.scheduler, config.train)

assert velocity_net_cfg.context_in_channels == 4, f"Got {velocity_net_cfg.context_in_channels}"
assert velocity_net_cfg.num_context_frames  == 3, f"Got {velocity_net_cfg.num_context_frames}"
print(f"velocity_net_cfg OK: context_in_channels={velocity_net_cfg.context_in_channels}, num_context_frames={velocity_net_cfg.num_context_frames}")

velocity_net_cfg OK: context_in_channels=4, num_context_frames=3


In [25]:
train_cfg = config.train

# Stochastic augmentations — train split only
augmentation = (
    compose_augmentation(*train_cfg.augmentations)
    if train_cfg.augmentations else None
)

# Deterministic transformations — all splits
transformation = (
    compose_transformation(*train_cfg.transformations)
    if train_cfg.transformations else None
)

data_module = WellDataModuleLightning(
    #well_base_path    = config.path.well_base_path,
    #well_dataset      = config.path.well_dataset_name,
    #stats_path        = config.path.stats_path,
    well_base_path    = '/Users/gabrieleaccarino/Workspace/waveletflux/data/datasets',
    well_dataset      = 'rayleigh_benard',
    stats_path        = '/Users/gabrieleaccarino/Workspace/waveletflux/data/datasets/rayleigh_benard/stats.yaml',
    n_steps_input     = config.train.n_steps_input,
    n_steps_output    = config.train.n_steps_output,
    max_rollout_steps = config.train.max_rollout_steps,
    batch_size        = config.train.batch_size,
    num_workers       = 0,
    train_shuffle     = train_cfg.train_shuffle,
    val_shuffle       = train_cfg.val_shuffle,
    test_shuffle      = train_cfg.test_shuffle,
    # rollout_val/test_shuffle not in TOML, default to False
    target_dtype      = eval(train_cfg.target_dtype),
    augmentation      = augmentation,
    transformation    = transformation,
)

data_module.setup()

normalizer  = data_module.normalizer
data_shape  = data_module.train_dataset.metadata.spatial_resolution
meta_fields = data_module.train_dataset.metadata.field_names
field_names = [item for sublist in meta_fields.values() for item in sublist]

In [26]:
model = FourierFlowModule(
    in_channels       = len(field_names),
    field_names       = field_names,
    data_shape        = data_shape,
    n_steps_output    = train_cfg.n_steps_output,
    scale_weights     = config.loss.scale_weights,
    subband_weights   = config.loss.get('subband_weights', None),
    fourier_cfg       = fourier_cfg,
    velocity_net_cfg  = velocity_net_cfg,
    flow_matching_cfg = flow_matching_cfg,
    optimizer_cfg     = optimizer_cfg,
    normalizer        = normalizer,
)

model.experiment_manager = None
callbacks = []

In [27]:
gradient_clip_val = lt_train.gradient_clip_val
gradient_clip_val = gradient_clip_val if isinstance(gradient_clip_val, float) else None

trainer = pl.Trainer(
    accelerator          = 'mps',
    strategy             = 'auto',
    devices              = 1,
    num_nodes            = 1,
    precision            = lt_train.precision,
    callbacks            = callbacks,
    max_epochs           = 1,
    logger               = False,
    gradient_clip_val    = gradient_clip_val,
    log_every_n_steps    = lt_train.log_every_n_steps,
    num_sanity_val_steps = 0,
    limit_train_batches  = lt_train.limit_train_batches,
    limit_val_batches    = lt_train.limit_val_batches,
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
`Trainer(limit_train_batches=1.0)` was configured so 100% of the batches per epoch will be used..
`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..


In [28]:
trainer.fit(model, datamodule=data_module)

/opt/anaconda3/envs/diffusion_env/lib/python3.12/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:751: Checkpoint directory /Users/gabrieleaccarino/Workspace/waveletflux/notebooks/checkpoints exists and is not empty.

  | Name               | Type                | Params | Mode 
-------------------------------------------------------------------
0 | fourier_decomposer | FourierDecomposer2D | 0      | train
1 | net                | WaveletFlowUNet     | 30.3 M | train
-------------------------------------------------------------------
30.3 M    Trainable params
0         Non-trainable params
30.3 M    Total params
121.177   Total estimated model params size (MB)
212       Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=1` reached.
